# A5 — Phân tích đã khóa và Explainable AI

Phân tích dự đoán từ A4 trên **tập test đã khóa**. A5 không huấn luyện lại hay lựa chọn mô hình từ test.

## 1. Chọn nguồn A4 đã chạy trên CSV thật

Cell đầu tự tìm component từ repository root, component hoặc notebooks directory. Không dùng smoke hoặc demo synthetic để viết báo cáo.

In [ ]:
from pathlib import Path
# Resolve the nested component from repository, component or notebooks directory.
_here = Path.cwd().resolve()
ROOT = None
for _base in (_here, *_here.parents):
    for _candidate in (_base, _base / 'research' / 'lead_modeling' / 'a4_a5'):
        if (_candidate / 'scripts' / 'run_a4.py').is_file() and (_candidate / 'config' / 'A2_protocol_v2.yaml').is_file():
            ROOT = _candidate
            break
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository, A4/A5 component or notebooks directory')
DATA=ROOT/'data'/'diabetes_binary_health_indicators_BRFSS2015.csv'
A4=ROOT/'results'/'a4_full_001'
OUT=ROOT/'results'/'a5_full_001'
assert (A4/'manifest.json').is_file()
import json
manifest=json.loads((A4/'manifest.json').read_text())
assert manifest['status']=='RESEARCH_TEST_COMPLETE', 'Không dùng smoke/demo làm kết quả'
manifest['status']

## 2. Phân tích thống kê + bootstrap 1.000 lần

In [ ]:
import subprocess,sys
cmd=[sys.executable,str(ROOT/'scripts'/'run_a5.py'),
     '--data',str(DATA),'--a4-dir',str(A4),'--output-dir',str(OUT),'--bootstraps','1000']
subprocess.run(cmd,check=True)

## 3. Kiểm tra bảng số liệu và đồ thị

Tất cả là kết quả từ code sau khi thực sự chạy với CSV Kaggle.

In [ ]:
import pandas as pd
from IPython.display import display,Image
display(pd.read_csv(A4/'test_metrics.csv'))
print(json.dumps(json.loads((OUT/'paired_bootstrap_summary.json').read_text()),indent=2,ensure_ascii=False))
display(Image(filename=str(OUT/'precision_recall_test.png')))
display(Image(filename=str(OUT/'confusion_matrix_final.png')))

## 4. Explainability (tùy chọn)

Cài `requirements-explain.txt` nếu cần SHAP/LIME. Sử dụng thư mục kết quả khác. Chỉ diễn giải hành vi mô hình, không suy ra nguyên nhân y tế.

In [ ]:
# Bỏ comment khi đã cài shap/lime và muốn phân tích thêm:
# XAI_OUT=ROOT/'results'/'a5_full_with_xai_001'
# cmd=[sys.executable,str(ROOT/'scripts'/'run_a5.py'),
#      '--data',str(DATA),'--a4-dir',str(A4),'--output-dir',str(XAI_OUT),
#      '--bootstraps','1000','--shap','--lime']
# subprocess.run(cmd,check=True)

## 5. Viết Discussion

Dùng `docs/A5_Results_Discussion_Template.md` và file Word A5 để điền số liệu thật, nêu cả kết quả âm/giới hạn. Ghi lại các tương tác AI và cách kiểm chứng.